# Step 4: Model Training

In this notebook, we train three different classifiers to predict NIFTY 50's daily direction (UP vs DOWN):
1. **Logistic Regression** (Interpretable baseline)
2. **Random Forest** (Non-linear bagging ensemble)
3. **Gradient Boosting** (Sequential boosting ensemble)

Since this is time-series data, we perform a chronological train-test split (no shuffling) and save the trained models, scaler, and predictions on the test dataset.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
import joblib

In [2]:
df = pd.read_csv("../data/processed/feature_engineered_data.csv")
print(f"Loaded feature-engineered dataset shape: {df.shape}")
df.head()

Loaded feature-engineered dataset shape: (3878, 31)


,Date,NIFTY_Close,NIFTY_High,NIFTY_Low,NIFTY_Open,Volume,Gold,Oil,USD_INR,India_VIX,...,Price_MA50_Diff,Momentum_3,Momentum_7,Volatility,Gold_Return,Oil_Return,USD_Return,VIX_Change,Future_Close,Target
0,2010-03-17,5231.899902,5260.500000,5177.149902,5198.450195,0,1124.000000,82.930000,45.279999,17.730000,...,202.948896,94.899902,107.899902,0.007403,0.001604,0.015055,-0.002423,-0.094946,5245.899902,1
1,2010-03-18,5245.899902,5255.649902,5214.399902,5232.549805,0,1127.400024,82.199997,45.389999,17.969999,...,216.674902,117.000000,144.399902,0.007146,0.003025,-0.008803,0.002429,0.013536,5262.799805,1
2,2010-03-19,5262.799805,5269.950195,5237.100098,5246.799805,0,1107.400024,80.680000,45.250000,17.750000,...,233.876807,64.699707,146.549805,0.006852,-0.017740,-0.018491,-0.003084,-0.012243,5205.200195,0
3,2010-03-22,5205.200195,5260.950195,5187.049805,5260.950195,0,1099.300049,81.250000,45.470001,19.959999,...,177.809189,-26.699707,71.800293,0.007075,-0.007314,0.007065,0.004862,0.124507,5225.299805,1
4,2010-03-23,5225.299805,5243.600098,5193.399902,5205.850098,0,1103.500000,81.910004,45.400002,18.129999,...,198.664805,-20.600098,88.299805,0.007069,0.003821,0.008123,-0.001539,-0.091683,5260.399902,1


In [3]:
# Define features and target
X = df.drop(columns=["Date", "Target", "Future_Close"])
y = df["Target"]
print(f"Features (X) Shape: {X.shape}")
print(f"Target (y) Shape: {y.shape}")
X.head()

Features (X) Shape: (3878, 28)
Target (y) Shape: (3878,)


,NIFTY_Close,NIFTY_High,NIFTY_Low,NIFTY_Open,Volume,Gold,Oil,USD_INR,India_VIX,Return,...,MA50,Price_MA20_Diff,Price_MA50_Diff,Momentum_3,Momentum_7,Volatility,Gold_Return,Oil_Return,USD_Return,VIX_Change
0,5231.899902,5260.500000,5177.149902,5198.450195,0,1124.000000,82.930000,45.279999,17.730000,0.006502,...,5028.951006,208.957422,202.948896,94.899902,107.899902,0.007403,0.001604,0.015055,-0.002423,-0.094946
1,5245.899902,5255.649902,5214.399902,5232.549805,0,1127.400024,82.199997,45.389999,17.969999,0.002676,...,5029.225000,206.362427,216.674902,117.000000,144.399902,0.007146,0.003025,-0.008803,0.002429,0.013536
2,5262.799805,5269.950195,5237.100098,5246.799805,0,1107.400024,80.680000,45.250000,17.750000,0.003222,...,5028.922998,204.509839,233.876807,64.699707,146.549805,0.006852,-0.017740,-0.018491,-0.003084,-0.012243
3,5205.200195,5260.950195,5187.049805,5260.950195,0,1099.300049,81.250000,45.470001,19.959999,-0.010945,...,5027.391006,128.895215,177.809189,-26.699707,71.800293,0.007075,-0.007314,0.007065,0.004862,0.124507
4,5225.299805,5243.600098,5193.399902,5205.850098,0,1103.500000,81.910004,45.400002,18.129999,0.003861,...,5026.635000,130.549829,198.664805,-20.600098,88.299805,0.007069,0.003821,0.008123,-0.001539,-0.091683


In [4]:
# Chronological Train-Test Split (no shuffling for time-series data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)
print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size: {X_test.shape[0]} rows")

Train size: 3102 rows
Test size: 776 rows


In [5]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")
print("Scaler fitted and saved successfully.")

Scaler fitted and saved successfully.


In [6]:
# Model 1: Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr, "../models/logistic_regression.pkl")
print("Logistic Regression model trained and saved.")

Training Logistic Regression...
Logistic Regression model trained and saved.


In [7]:
# Model 2: Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train_scaled, y_train) # Using scaled inputs consistently for all models
joblib.dump(rf, "../models/random_forest.pkl")
print("Random Forest model trained and saved.")

Training Random Forest...


Random Forest model trained and saved.


In [8]:
# Model 3: Gradient Boosting
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_train) # Using scaled inputs consistently for all models
joblib.dump(gb, "../models/gradient_boosting.pkl")
print("Gradient Boosting model trained and saved.")

Training Gradient Boosting...


Gradient Boosting model trained and saved.


In [9]:
# Generate Predictions
lr_pred = lr.predict(X_test_scaled)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]

rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]

gb_pred = gb.predict(X_test_scaled)
gb_prob = gb.predict_proba(X_test_scaled)[:, 1]

print("Example probability output for Logistic Regression:")
print(lr_prob[:5])

Example probability output for Logistic Regression:
[0.57391528 0.58676181 0.60700741 0.55423017 0.54748922]


In [10]:
# Save Predictions to CSV
predictions = pd.DataFrame({
    "Actual": y_test,
    "LR": lr_pred,
    "LR_Prob": lr_prob,
    "RF": rf_pred,
    "RF_Prob": rf_prob,
    "GB": gb_pred,
    "GB_Prob": gb_prob
})

predictions.to_csv("../data/processed/model_predictions.csv", index=False)
print("Saved model predictions and probabilities to data/processed/model_predictions.csv")

Saved model predictions and probabilities to data/processed/model_predictions.csv
